In [1]:
#version 1 - it jst finds the top 10 tf-idf words from the articel selected and displays score too.
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

article_file_path = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\us_newsarticle_August_1"

#this part loads the dataset,perform pre-processing and splits all the articles into a list
def load_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    raw_articles = content.split('**************')
    
    articles_data = []
    for art in raw_articles:
        if art.strip():
            lines = art.strip().split('\n')
            title_match = re.search(r'-----(.*?)-----', lines[0])
            title = title_match.group(1).strip() if title_match else "Untitled"
            
            body = " ".join(lines[1:]).strip()
            articles_data.append({"Title": title, "Content": body})
            
    return pd.DataFrame(articles_data)
# here we do re to prepocess
def preprocess_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text) 
    text = re.sub(r'\s+', ' ', text).strip() 
    return text
df = load_dataset(article_file_path)
df['Cleaned_Content'] = df['Content'].apply(preprocess_text)

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Content'])
feature_names = vectorizer.get_feature_names_out()
# we choose the desired article 
print("Articles found in the dataset:")
for i, title in enumerate(df['Title']):
    print(f"{i+1}. {title}")

user_input = input("\nEnter the Article Number or Exact Title to analyze: ")

try:
    if user_input.isdigit():
        idx = int(user_input) - 1
    else:
        idx = df[df['Title'].str.lower() == user_input.lower()].index[0]

    article_vector = tfidf_matrix[idx]
    df_tfidf = pd.DataFrame(article_vector.T.todense(), index=feature_names, columns=["score"])
    
    top_10 = df_tfidf.sort_values(by="score", ascending=False).head(10)

    print(f"\nTop 10 TF-IDF Terms for: {df['Title'].iloc[idx]}")
    print("-" * 30)
    print(top_10)

except (IndexError, ValueError):
    print("Invalid selection. Please ensure you enter a valid number or match the title exactly.")

Articles found in the dataset:
1. Melania Trump's girl-on-girl photos from racy shoot revealed
2. Ex-Venezuelan narcotics officials face drug-trafficking charges
3. Props used in iconic movies to hit the auction block
4. Zika outbreak prompts travel warning for parts of Miami
5. Iranians outraged after porn star visits for nose job
6. Warren Buffett joins the billionaires-for-Hillary club
7. Constitution sales soar after soldier's dad displayed his at DNC
8. Armed trio stopped by cops trying to stock up at Walmart for 'Doomsday'
9. Prankster cops hand out ice cream cones instead of tickets
10. FBI electronics technician charged with spying for China
11. America launches airstrikes against ISIS in Libya
12. Florida seeks federal emergency response to Zika outbreak
13. Texas college students can now go to class armed
14. Corey Lewandowski freaks out when Christine Quinn touches him
15. Newly crowned Miss Teen USA apologizes for racist tweets
16. Clinton takes 7-point lead over Trump in p


Enter the Article Number or Exact Title to analyze:  3



Top 10 TF-IDF Terms for: Props used in iconic movies to hit the auction block
------------------------------
             score
auction   0.306492
movie     0.272342
rake      0.153246
prop      0.153246
jaws      0.153246
store     0.153246
film      0.153246
items     0.153246
worn      0.153246
expected  0.136171


In [1]:
#v2 here we extract the top 10 tf idf words and based on thoose words we processa all the 
# tweet files in the All_tweets folder and it finds relevant tweets only based on the presence of 
# the word
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer


ARTICLE_PATH = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\us_newsarticle_August_1"
TWEETS_FOLDER = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\ALL_TWEETS"

def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    raw_articles = content.split('**************')

    articles_data = []
    for art in raw_articles:
        if art.strip():
            lines = art.strip().split('\n')

            title_match = re.search(r'-----(.*?)-----', lines[0])
            title = title_match.group(1).strip() if title_match else "Untitled"

            body = " ".join(lines[1:]).strip()

            articles_data.append({
                "Title": title,
                "Content": body
            })

    return pd.DataFrame(articles_data)


def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = load_dataset(ARTICLE_PATH)
df['Cleaned_Content'] = df['Content'].apply(preprocess_text)


vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Content'])
feature_names = vectorizer.get_feature_names_out()


print("Articles found in the dataset:")
for i, title in enumerate(df['Title']):
    print(f"{i+1}. {title}")

user_input = input("\nEnter the Article Number to analyze: ")

try:
    idx = int(user_input) - 1

    article_vector = tfidf_matrix[idx]

    df_tfidf = pd.DataFrame(
        article_vector.T.todense(),
        index=feature_names,
        columns=["score"]
    )

    top_10 = df_tfidf.sort_values(by="score", ascending=False).head(10)

    keyword_weights = top_10['score'].to_dict()

    print(f"\nTop 10 Keywords identified: {list(keyword_weights.keys())}")

    all_relevant_tweets = []

    def process_tweet_file(file_path):
        rows = []
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = line.split('*****', 1)
                if len(parts) != 2:
                    continue

                tweet_id = parts[0].strip()
                tweet_text = parts[1].strip()

                rows.append((tweet_id, tweet_text))

        return rows

    def calculate_relevance(text):
        if not isinstance(text, str):
            return 0

        words = set(preprocess_text(text).split())
        score = 0

        for word, weight in keyword_weights.items():
            if word in words:
                score += weight

        return score

    for filename in os.listdir(TWEETS_FOLDER):
        file_path = os.path.join(TWEETS_FOLDER, filename)

        if os.path.isfile(file_path):
            print(f"Processing file: {filename}")

            tweets = process_tweet_file(file_path)

            for tweet_id, tweet_text in tweets:
                score = calculate_relevance(tweet_text)

                if score > 0:
                    all_relevant_tweets.append({
                        "tweet_id": tweet_id,
                        "tweet_text": tweet_text,
                        "relevance_score": score,
                        "source_file": filename
                    })

    relevant_df = pd.DataFrame(all_relevant_tweets)

    if not relevant_df.empty:
        relevant_df = relevant_df.sort_values(by='relevance_score', ascending=False)

    output_file = f"relevant_tweets_Article_{idx+1}.csv"
    relevant_df.to_csv(output_file, index=False)

    print(f"\n Saved {len(relevant_df)} relevant tweets to {output_file}")

except Exception as e:
    print("Error occurred:", e)

Articles found in the dataset:
1. Melania Trump's girl-on-girl photos from racy shoot revealed
2. Ex-Venezuelan narcotics officials face drug-trafficking charges
3. Props used in iconic movies to hit the auction block
4. Zika outbreak prompts travel warning for parts of Miami
5. Iranians outraged after porn star visits for nose job
6. Warren Buffett joins the billionaires-for-Hillary club
7. Constitution sales soar after soldier's dad displayed his at DNC
8. Armed trio stopped by cops trying to stock up at Walmart for 'Doomsday'
9. Prankster cops hand out ice cream cones instead of tickets
10. FBI electronics technician charged with spying for China
11. America launches airstrikes against ISIS in Libya
12. Florida seeks federal emergency response to Zika outbreak
13. Texas college students can now go to class armed
14. Corey Lewandowski freaks out when Christine Quinn touches him
15. Newly crowned Miss Teen USA apologizes for racist tweets
16. Clinton takes 7-point lead over Trump in p


Enter the Article Number to analyze:  1



Top 10 Keywords identified: ['melania', 'photo', 'basseville', 'beauty', 'eriksson', 'featured', 'shoot', 'fashion', 'source', 'pictures']
Processing file: all_Aug_1_tweets_0
Processing file: all_Aug_1_tweets_1
Processing file: all_Aug_1_tweets_10
Processing file: all_Aug_1_tweets_11
Processing file: all_Aug_1_tweets_12
Processing file: all_Aug_1_tweets_13
Processing file: all_Aug_1_tweets_14
Processing file: all_Aug_1_tweets_15
Processing file: all_Aug_1_tweets_16
Processing file: all_Aug_1_tweets_17
Processing file: all_Aug_1_tweets_18
Processing file: all_Aug_1_tweets_19
Processing file: all_Aug_1_tweets_2
Processing file: all_Aug_1_tweets_20
Processing file: all_Aug_1_tweets_21
Processing file: all_Aug_1_tweets_22
Processing file: all_Aug_1_tweets_23
Processing file: all_Aug_1_tweets_24
Processing file: all_Aug_1_tweets_25
Processing file: all_Aug_1_tweets_26
Processing file: all_Aug_1_tweets_27
Processing file: all_Aug_1_tweets_28
Processing file: all_Aug_1_tweets_3
Processing fi

In [2]:
#v3 here we remove the duplicate tweetd which have the save tweet id ,and save only the unique relevant tweets
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer


ARTICLE_PATH = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\us_newsarticle_August_1"
TWEETS_FOLDER = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\ALL_TWEETS"

def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    raw_articles = content.split('**************')

    articles_data = []
    for art in raw_articles:
        if art.strip():
            lines = art.strip().split('\n')

            title_match = re.search(r'-----(.*?)-----', lines[0])
            title = title_match.group(1).strip() if title_match else "Untitled"

            body = " ".join(lines[1:]).strip()

            articles_data.append({
                "Title": title,
                "Content": body
            })

    return pd.DataFrame(articles_data)


def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = load_dataset(ARTICLE_PATH)
df['Cleaned_Content'] = df['Content'].apply(preprocess_text)

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Content'])
feature_names = vectorizer.get_feature_names_out()

print("Articles found in the dataset:")
for i, title in enumerate(df['Title']):
    print(f"{i+1}. {title}")

user_input = input("\nEnter the Article Number to analyze: ")

try:
    idx = int(user_input) - 1
    article_vector = tfidf_matrix[idx]

    df_tfidf = pd.DataFrame(
        article_vector.T.todense(),
        index=feature_names,
        columns=["score"]
    )

    top_10 = df_tfidf.sort_values(by="score", ascending=False).head(10)
    keyword_weights = top_10['score'].to_dict()

    print(f"\nTop 10 Keywords identified: {list(keyword_weights.keys())}")

    main_keyword = list(keyword_weights.keys())[0]

    all_relevant_tweets = []

    def process_tweet_file(file_path):
        rows = []
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = line.split('*****', 1)
                if len(parts) != 2:
                    continue

                tweet_id = parts[0].strip()
                tweet_text = parts[1].strip()

                rows.append((tweet_id, tweet_text))

        return rows

    def calculate_relevance(text):
        if not isinstance(text, str):
            return 0

        clean_text = preprocess_text(text)
        score = 0

        for word, weight in keyword_weights.items():
            if word in clean_text:
                score += weight

        return score

    for filename in os.listdir(TWEETS_FOLDER):
        file_path = os.path.join(TWEETS_FOLDER, filename)

        if os.path.isfile(file_path):
            print(f"Processing file: {filename}")

            tweets = process_tweet_file(file_path)

            for tweet_id, tweet_text in tweets:
                clean_text = preprocess_text(tweet_text)

                score = calculate_relevance(tweet_text)
                if score > 0 and main_keyword in clean_text:
                    all_relevant_tweets.append({
                        "article_id": idx + 1,
                        "article_title": df['Title'].iloc[idx],
                        "tweet_id": tweet_id,
                        "tweet_text": tweet_text,
                        "relevance_score": score,
                        "source_file": filename
                    })

    relevant_df = pd.DataFrame(all_relevant_tweets)

    if not relevant_df.empty:
        relevant_df = relevant_df.sort_values(by='relevance_score', ascending=False)

    output_file = f"relevant_tweets_New_Article_{idx+1}.csv"
    relevant_df.to_csv(output_file, index=False)

    print(f"\n Saved {len(relevant_df)} relevant tweets to {output_file}")

except Exception as e:
    print(" Error occurred:", e)

Articles found in the dataset:
1. Melania Trump's girl-on-girl photos from racy shoot revealed
2. Ex-Venezuelan narcotics officials face drug-trafficking charges
3. Props used in iconic movies to hit the auction block
4. Zika outbreak prompts travel warning for parts of Miami
5. Iranians outraged after porn star visits for nose job
6. Warren Buffett joins the billionaires-for-Hillary club
7. Constitution sales soar after soldier's dad displayed his at DNC
8. Armed trio stopped by cops trying to stock up at Walmart for 'Doomsday'
9. Prankster cops hand out ice cream cones instead of tickets
10. FBI electronics technician charged with spying for China
11. America launches airstrikes against ISIS in Libya
12. Florida seeks federal emergency response to Zika outbreak
13. Texas college students can now go to class armed
14. Corey Lewandowski freaks out when Christine Quinn touches him
15. Newly crowned Miss Teen USA apologizes for racist tweets
16. Clinton takes 7-point lead over Trump in p


Enter the Article Number to analyze:  1



Top 10 Keywords identified: ['melania', 'photo', 'basseville', 'beauty', 'eriksson', 'featured', 'shoot', 'fashion', 'source', 'pictures']
Processing file: all_Aug_1_tweets_0
Processing file: all_Aug_1_tweets_1
Processing file: all_Aug_1_tweets_10
Processing file: all_Aug_1_tweets_11
Processing file: all_Aug_1_tweets_12
Processing file: all_Aug_1_tweets_13
Processing file: all_Aug_1_tweets_14
Processing file: all_Aug_1_tweets_15
Processing file: all_Aug_1_tweets_16
Processing file: all_Aug_1_tweets_17
Processing file: all_Aug_1_tweets_18
Processing file: all_Aug_1_tweets_19
Processing file: all_Aug_1_tweets_2
Processing file: all_Aug_1_tweets_20
Processing file: all_Aug_1_tweets_21
Processing file: all_Aug_1_tweets_22
Processing file: all_Aug_1_tweets_23
Processing file: all_Aug_1_tweets_24
Processing file: all_Aug_1_tweets_25
Processing file: all_Aug_1_tweets_26
Processing file: all_Aug_1_tweets_27
Processing file: all_Aug_1_tweets_28
Processing file: all_Aug_1_tweets_3
Processing fi

In [3]:
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

ARTICLE_PATH = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\us_newsarticle_August_1"
TWEETS_FOLDER = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\ALL_TWEETS"


def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    raw_articles = content.split('**************')

    articles_data = []
    for art in raw_articles:
        if art.strip():
            lines = art.strip().split('\n')

            title_match = re.search(r'-----(.*?)-----', lines[0])
            title = title_match.group(1).strip() if title_match else "Untitled"

            body = " ".join(lines[1:]).strip()

            articles_data.append({
                "Title": title,
                "Content": body
            })

    return pd.DataFrame(articles_data)

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = load_dataset(ARTICLE_PATH)
df['Cleaned_Content'] = df['Content'].apply(preprocess_text)

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Content'])
feature_names = vectorizer.get_feature_names_out()

print("Articles found in the dataset:")
for i, title in enumerate(df['Title']):
    print(f"{i+1}. {title}")

user_input = input("\nEnter the Article Number to analyze: ")

try:
    idx = int(user_input) - 1
    article_vector = tfidf_matrix[idx]

    df_tfidf = pd.DataFrame(
        article_vector.T.todense(),
        index=feature_names,
        columns=["score"]
    )

    top_10 = df_tfidf.sort_values(by="score", ascending=False).head(10)
    keyword_weights = top_10['score'].to_dict()

    print(f"\nTop 10 Keywords identified: {list(keyword_weights.keys())}")

    main_keyword = list(keyword_weights.keys())[0]
    all_relevant_tweets = []

    def process_tweet_file(file_path):
        rows = []
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = line.split('*****', 1)
                if len(parts) != 2:
                    continue

                tweet_id = parts[0].strip()
                tweet_text = parts[1].strip()

                rows.append((tweet_id, tweet_text))

        return rows

    def calculate_relevance(text):
        if not isinstance(text, str):
            return 0

        clean_text = preprocess_text(text)
        score = 0

        for word, weight in keyword_weights.items():
            if word in clean_text:
                score += weight

        return score

    for filename in os.listdir(TWEETS_FOLDER):
        file_path = os.path.join(TWEETS_FOLDER, filename)

        if os.path.isfile(file_path):
            print(f"Processing file: {filename}")

            tweets = process_tweet_file(file_path)

            for tweet_id, tweet_text in tweets:
                clean_text = preprocess_text(tweet_text)

                score = calculate_relevance(tweet_text)
                if score > 0 and main_keyword in clean_text:
                    all_relevant_tweets.append({
                        "article_id": idx + 1,
                        "article_title": df['Title'].iloc[idx],
                        "tweet_id": tweet_id,
                        "tweet_text": tweet_text,
                        "relevance_score": score,
                        "source_file": filename
                    })
    relevant_df = pd.DataFrame(all_relevant_tweets)

    if not relevant_df.empty:
        relevant_df['tweet_id'] = relevant_df['tweet_id'].astype(str)
        relevant_df = relevant_df.drop_duplicates(subset=['tweet_id'])
        relevant_df = relevant_df.sort_values(by='relevance_score', ascending=False)

    output_file = f"relevant_tweets_Article_Mod{idx+1}.csv"
    relevant_df.to_csv(output_file, index=False)

    print(f"\n Saved {len(relevant_df)} unique relevant tweets to {output_file}")

except Exception as e:
    print(" Error occurred:", e)

Articles found in the dataset:
1. Melania Trump's girl-on-girl photos from racy shoot revealed
2. Ex-Venezuelan narcotics officials face drug-trafficking charges
3. Props used in iconic movies to hit the auction block
4. Zika outbreak prompts travel warning for parts of Miami
5. Iranians outraged after porn star visits for nose job
6. Warren Buffett joins the billionaires-for-Hillary club
7. Constitution sales soar after soldier's dad displayed his at DNC
8. Armed trio stopped by cops trying to stock up at Walmart for 'Doomsday'
9. Prankster cops hand out ice cream cones instead of tickets
10. FBI electronics technician charged with spying for China
11. America launches airstrikes against ISIS in Libya
12. Florida seeks federal emergency response to Zika outbreak
13. Texas college students can now go to class armed
14. Corey Lewandowski freaks out when Christine Quinn touches him
15. Newly crowned Miss Teen USA apologizes for racist tweets
16. Clinton takes 7-point lead over Trump in p


Enter the Article Number to analyze:  1



Top 10 Keywords identified: ['melania', 'photo', 'basseville', 'beauty', 'eriksson', 'featured', 'shoot', 'fashion', 'source', 'pictures']
Processing file: all_Aug_1_tweets_0
Processing file: all_Aug_1_tweets_1
Processing file: all_Aug_1_tweets_10
Processing file: all_Aug_1_tweets_11
Processing file: all_Aug_1_tweets_12
Processing file: all_Aug_1_tweets_13
Processing file: all_Aug_1_tweets_14
Processing file: all_Aug_1_tweets_15
Processing file: all_Aug_1_tweets_16
Processing file: all_Aug_1_tweets_17
Processing file: all_Aug_1_tweets_18
Processing file: all_Aug_1_tweets_19
Processing file: all_Aug_1_tweets_2
Processing file: all_Aug_1_tweets_20
Processing file: all_Aug_1_tweets_21
Processing file: all_Aug_1_tweets_22
Processing file: all_Aug_1_tweets_23
Processing file: all_Aug_1_tweets_24
Processing file: all_Aug_1_tweets_25
Processing file: all_Aug_1_tweets_26
Processing file: all_Aug_1_tweets_27
Processing file: all_Aug_1_tweets_28
Processing file: all_Aug_1_tweets_3
Processing fi